# Chapter 14 — Working with PLFS Unit-Level Data

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch14_plfs_unit_level_data.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 14.1 The survey and its files

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from indiadata import read_plfs, layout_names, labour_status, labour_rates

PLFS = Path("data/plfs")
sorted(f.name for f in PLFS.iterdir())

## 14.2 Reading the raw files

In [ ]:
raw_header = pd.read_csv(PLFS / "perv1.csv", nrows=0).columns
print(len(raw_header), "columns")
print(list(raw_header[:20]))

In [ ]:
names = layout_names("perv1", folder=PLFS)
pd.DataFrame({"raw": raw_header, "field_name": names}).iloc[16:34]

In [ ]:
cols = ["sex", "age", "gedu_lvl", "pas", "sas", "acws", "ern_reg"]
p = read_plfs("perv1", folder=PLFS, columns=cols)

print(f"{len(p):,} persons in {p['mfsu'].nunique():,} first-stage units")
print(f"memory used: {p.memory_usage(deep=True).sum() / 1e6:.0f} MB")
p.head()

## 14.3 Codes, not words

In [ ]:
STATUS_LABELS = {
    11: "Own-account worker", 12: "Employer", 21: "Unpaid family helper",
    31: "Regular wage/salaried", 41: "Casual: public works",
    51: "Casual: other", 81: "Unemployed (sought work)",
    91: "Student", 92: "Domestic duties only",
    93: "Domestic duties + free collection", 94: "Rentier/pensioner",
    95: "Not able to work (disability)", 97: "Others", 99: "Child (age 0-4)",
}
counts = p["pas"].value_counts().sort_index()
pd.DataFrame({"label": counts.index.map(STATUS_LABELS), "records": counts})

In [ ]:
p["sector"] = p["sec"].map({1: "Rural", 2: "Urban"})
p["gender"] = p["sex"].map({1: "Male", 2: "Female", 3: "Transgender"})
p["gender"].value_counts()

## 14.4 Survey weights: why every row is not equal

In [ ]:
share_sample = (p["sector"] == "Urban").mean()
print(f"Urban share of persons in the sample: {share_sample:.1%}")

In [ ]:
w = np.where(p["nss"] == p["nsc"], p["mult"] / 100, p["mult"] / 200) / p["no_qtr"]
print("Same as read_plfs weight:", np.allclose(w, p["weight"]))

pop = p["weight"].sum()
print(f"Persons represented: {pop / 1e7:.1f} crore")
share_weighted = p.loc[p["sector"] == "Urban", "weight"].sum() / pop
print(f"Urban share, weighted: {share_weighted:.1%}")

## 14.5 LFPR, WPR and UR, by hand and then with a function

In [ ]:
employed = p["pas"].between(11, 51) | p["sas"].between(11, 51)
unemployed = (p["pas"] == 81) & ~employed
adult = p["age"] >= 15

W = p["weight"]
pop15 = W[adult].sum()
emp15 = W[adult & employed].sum()
unemp15 = W[adult & unemployed].sum()

print(f"LFPR {100 * (emp15 + unemp15) / pop15:.1f}%")
print(f"WPR  {100 * emp15 / pop15:.1f}%")
print(f"UR   {100 * unemp15 / (emp15 + unemp15):.1f}%")

In [ ]:
p["status"] = labour_status(p["pas"], p["sas"])
a = p[p["age"] >= 15]                    # adults, used from here on
headline = labour_rates(a)
headline.round(1)

In [ ]:
published = pd.Series({"LFPR": 60.1, "WPR": 58.2, "UR": 3.2})
check = pd.DataFrame({"ours": headline.round(1), "MoSPI": published})
check["match"] = check["ours"] == check["MoSPI"]
check

In [ ]:
unweighted = (a["status"] != "out of LF").mean() * 100
print(f"Unweighted LFPR: {unweighted:.1f}%   weighted: {headline['LFPR']:.1f}%")

## 14.6 Rural and urban, men and women

In [ ]:
mf = a[a["sex"].isin([1, 2])]
by_group = mf.groupby(["sector", "gender"]).apply(labour_rates).round(1)
by_group

In [ ]:
lfpr = by_group["LFPR"].unstack("gender")[["Male", "Female"]]

fig, ax = plt.subplots(figsize=(7, 4))
lfpr.plot.bar(ax=ax, color=["#1f4e79", "#c55a11"], width=0.7, rot=0)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=9)
ax.set_ylabel("LFPR, % of persons aged 15+")
ax.set_xlabel("")
ax.set_ylim(0, 95)
ax.set_title("Labour force participation, 2023–24 (usual status)",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.text(0.01, -0.03, "Source: author's calculation from PLFS 2023–24 "
         "unit-level data (NSO, MoSPI), weighted.", fontsize=8, color="grey")
plt.show()

## 14.7 Participation over the life cycle

In [ ]:
mf = mf.assign(age_band=(mf["age"] // 5 * 5).clip(upper=75))
life = (mf.groupby(["age_band", "gender"]).apply(labour_rates)["LFPR"]
          .unstack("gender")[["Male", "Female"]])

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(life.index, life["Male"], marker="o", color="#1f4e79", label="Male")
ax.plot(life.index, life["Female"], marker="o", color="#c55a11", label="Female")
ax.set_xticks(life.index)
ax.set_xticklabels([f"{b}–{b + 4}" if b < 75 else "75+" for b in life.index],
                   rotation=45)
ax.set_ylabel("LFPR, %")
ax.set_ylim(0, 105)
ax.set_title("LFPR by age, 2023–24 (usual status)", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.text(0.01, -0.08, "Source: author's calculation from PLFS 2023–24 "
         "unit-level data (NSO, MoSPI), weighted.", fontsize=8, color="grey")
plt.show()

In [ ]:
peak_f = life["Female"].idxmax()
print(f"Men: LFPR above 95% from age {life.index[life['Male'] > 95].min()} "
      f"to {life.index[life['Male'] > 95].max() + 4}")
print(f"Women: peak LFPR {life['Female'].max():.1f}% at ages "
      f"{peak_f}–{peak_f + 4}")

## 14.8 States: where do women work?

In [ ]:
layout_file = next(PLFS.glob("Data_Layout*.xlsx"))
states = pd.read_excel(layout_file, sheet_name="State code", header=1)
states.columns = ["st", "state"]
states = states.dropna().astype({"st": int})
STATE = dict(zip(states["st"], states["state"].str.strip()))

female = a[a["sex"] == 2]
f_state = female.groupby("st").apply(labour_rates)
f_state.index = f_state.index.map(STATE)
f_state = f_state.sort_values("LFPR")
f_state[["LFPR", "UR"]].round(1).T

In [ ]:
headline_f = labour_rates(female)["LFPR"]

fig, ax = plt.subplots(figsize=(7, 9))
colors = ["#c55a11" if s == "Maharashtra" else "#9db7d5" for s in f_state.index]
ax.barh(f_state.index, f_state["LFPR"], color=colors)
ax.axvline(headline_f, color="black", lw=1, ls="--")
ax.text(headline_f + 0.5, 0.2, f"All India {headline_f:.1f}%", fontsize=9)
ax.set_xlabel("Female LFPR, % of women aged 15+")
ax.set_title("Female labour force participation by state, 2023–24",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", labelsize=8)
fig.text(0.01, 0.005, "Usual status (ps+ss). Small states and UTs have small "
         "samples; treat their estimates with care.\nSource: author's "
         "calculation from PLFS 2023–24 unit-level data (NSO, MoSPI).",
         fontsize=8, color="grey")
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

In [ ]:
top, bottom = f_state.index[-1], f_state.index[0]
print(f"Highest: {top} ({f_state.loc[top, 'LFPR']:.1f}%)")
print(f"Lowest:  {bottom} ({f_state.loc[bottom, 'LFPR']:.1f}%)")
print(f"Maharashtra: {f_state.loc['Maharashtra', 'LFPR']:.1f}%")

## 14.9 Educated unemployment among the young

In [ ]:
youth = a[a["age"].between(15, 29)]
print(f"Youth (15–29) unemployment rate: {labour_rates(youth)['UR']:.1f}%")

In [ ]:
EDU = {1: "Not literate", 2: "Up to primary", 3: "Up to primary",
       4: "Up to primary", 5: "Up to primary", 6: "Up to primary",
       7: "Middle", 8: "Secondary", 10: "Higher secondary",
       11: "Diploma/certificate", 12: "Graduate", 13: "Postgraduate+"}
ORDER = ["Not literate", "Up to primary", "Middle", "Secondary",
         "Higher secondary", "Diploma/certificate", "Graduate", "Postgraduate+"]

youth = youth[youth["sex"].isin([1, 2])].assign(edu=youth["gedu_lvl"].map(EDU))
ur_edu = (youth.groupby(["edu", "gender"]).apply(labour_rates)["UR"]
               .unstack("gender").loc[ORDER, ["Male", "Female"]])
n_lf = (youth[youth["status"] != "out of LF"]
        .groupby(["edu", "gender"]).size().unstack("gender")
        .loc[ORDER, ["Male", "Female"]])
pd.concat({"UR %": ur_edu.round(1), "sample in labour force": n_lf}, axis=1)

In [ ]:
smallest = n_lf.stack().idxmin()
print(f"Smallest cell: {smallest[1].lower()} youth with {smallest[0].lower()} "
      f"education, {n_lf.stack().min():,} persons in the labour force")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ur_edu.plot.bar(ax=ax, color=["#1f4e79", "#c55a11"], width=0.75, rot=30)
ax.set_ylabel("Unemployment rate, %")
ax.set_xlabel("")
ax.set_title("Youth (15–29) unemployment by education, 2023–24",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.setp(ax.get_xticklabels(), ha="right")
fig.text(0.01, -0.12, "Usual status (ps+ss). Source: author's calculation "
         "from PLFS 2023–24 unit-level data (NSO, MoSPI).",
         fontsize=8, color="grey")
plt.show()

## 14.10 A different lens: current weekly status

In [ ]:
p["status_cws"] = labour_status(p["acws"], weekly=True)
print("All ages, CWS:", labour_rates(p, status="status_cws").round(1).to_dict())

a = p[p["age"] >= 15]
compare = pd.DataFrame({
    "Usual status (ps+ss)": labour_rates(a),
    "Current weekly status": labour_rates(a, status="status_cws"),
}).round(1)
compare

## 14.11 How precise are these numbers?

In [ ]:
a = a.assign(w_sub=a["mult"] / 100 / a["no_qtr"])


def subsample_se(df, status="status"):
    """Rates from each PLFS sub-sample and the half-difference standard error."""
    est = pd.DataFrame({k: labour_rates(df[df["ss"] == k], status, "w_sub")
                        for k in (1, 2)})
    est.columns = ["sub-sample 1", "sub-sample 2"]
    est["full sample"] = labour_rates(df, status)
    est["approx. SE"] = (est["sub-sample 1"] - est["sub-sample 2"]).abs() / 2
    return est.round(2)


print("All India, persons 15+")
display(subsample_se(a))
print("Women 15+, Maharashtra")
display(subsample_se(a[(a["sex"] == 2) & (a["st"] == 27)]))

## 14.12 What do salaried workers earn?

In [ ]:
def weighted_median(x, w):
    """Value at which cumulative weight first reaches half the total."""
    x, w = x.to_numpy(dtype=float), w.to_numpy(dtype=float)
    order = np.argsort(x)
    x, w = x[order], w[order]
    cum = np.cumsum(w) / w.sum()
    return x[np.searchsorted(cum, 0.5)]


salaried = p[(p["acws"] == 31) & (p["ern_reg"] > 0)
             & p["age"].between(15, 59) & p["sex"].isin([1, 2])]
earn = salaried.groupby(["sector", "gender"]).apply(
    lambda d: pd.Series({
        "median (₹/month)": weighted_median(d["ern_reg"], d["weight"]),
        "mean (₹/month)": np.average(d["ern_reg"], weights=d["weight"]),
        "sample": len(d),
    })).round(0)
earn

In [ ]:
ratio = (earn["median (₹/month)"].xs("Female", level="gender")
         / earn["median (₹/month)"].xs("Male", level="gender"))
print("Women's median salary as a share of men's:")
print(ratio.map("{:.0%}".format).to_string())

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading